#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col
from pyspark.sql.window import Window

In [0]:
RENAME_MAP = {
    "cst_id" : "customer_id",
    "cst_key" : "customer_key",
    "cst_firstname" : "first_name",
    "cst_lastname" : "last_name",
    "cst_marital_status" : "marital_status",
    "cst_gndr" : "gender",
    "cst_create_date" : "create_date"
}

#Reading From Bronze

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")
df.display()

# Data Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Null handling and Empty values

In [0]:
def is_empty(col):
    return col.isNull() | (F.trim(col) == "")

condition_invalid = (
    is_empty(F.col("cst_firstname")) |
    is_empty(F.col("cst_lastname")) |
    F.col("cst_id").isNull()
)

df_valid = df.filter(~condition_invalid)
df_invalid = df.filter(condition_invalid)

invalid_count = df_invalid.count()
print("Invalid rows:", invalid_count)

df = df_valid

## Data type casting

In [0]:
df = df.withColumn("cst_create_date", F.to_date(F.col("cst_create_date")))

## Handle duplicate ids

In [0]:
dup_count = df.groupBy("cst_id").count().filter("count > 1").count()

print("Duplicate cst_id:", dup_count)

window = Window.partitionBy("cst_id").orderBy(F.col("cst_create_date").desc())

df = (
    df.withColumn("row_num", F.row_number().over(window))
      .filter(F.col("row_num") == 1)
      .drop("row_num")
)



## AWS format check

In [0]:
df_invalid_key = df.filter(
    ~F.col("cst_key").rlike("^AW[0-9]{8}$")
)

print("Invalid customer_key:", df_invalid_key.count())

df = df.filter(F.col("cst_key").rlike("^AW[0-9]{8}$"))

##Date validation

In [0]:
df = df.withColumn("cst_create_date", F.to_date("cst_create_date"))

invalid_dates = df.filter(F.col("cst_create_date") > F.current_date())

print("Invalid create_dates:", invalid_dates.count())

##Normalization

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise(None)
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .otherwise(None)
    )
)

## Renamig the columns

In [0]:
for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks before write

In [0]:
def sanity_check(df):
    row_count = df.count()

    duplicate_customer_id = (
        df.groupBy("customer_id")
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    null_critical_fields = (
        df.filter(
            F.col("customer_id").isNull() |
            F.col("first_name").isNull() |
            F.col("last_name").isNull()
        )
        .count()
    )

    invalid_customer_key = (
        df.filter(
            ~F.col("customer_key").rlike("^AW[0-9]{8}$")
        )
        .count()
    )

    future_dates = (
        df.filter(
            F.col("create_date") > F.current_date()
        )
        .count()
    )
    return {
        "row_count": row_count,
        "duplicate_customer_id": duplicate_customer_id,
        "null_critical_fields": null_critical_fields,
        "invalid_customer_key": invalid_customer_key,
        "future_dates": future_dates
    }

results = sanity_check(df)
print(results)

# Write Into Silver

In [0]:
(df.write.mode("overwrite").format("delta").saveAsTable("silver.crm_customers"))

df_silver = spark.table("workspace.silver.crm_customers")
sanity_check(df_silver)